### **Environment Configuration**

In this preliminary block, we set up the working environment by installing advanced libraries for Natural Language Processing (NLP):

1.  **Dependency Installation:** We install specific tools for semantic analysis: **VADER** (lexicon-based sentiment), **Transformers** (for Deep Learning models like BERT), **BERTopic** (for topic modeling), and **Sentence-Transformers** (for semantic similarity calculation).
2.  **Import:** We load essential modules for data management (`pandas`, `numpy`) and initialize the AI libraries just installed.

In [ ]:
!pip install -q vadersentiment transformers scipy bertopic sentence-transformers

import pandas as pd
import numpy as np
import re
import os
import torch
from tqdm.auto import tqdm
from google.colab import drive
from collections import Counter
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 8.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


### **Social Content Analysis (SCA)**

In this phase, the project shifts from structural analysis (who interacts with whom) to semantic analysis (what they talk about and how). The objective is to profile the communities identified in the SNA phase through their textual content.

The methodology combines traditional approaches (Lexicon-based) with Deep Learning models (Transformer-based) to ensure scientific robustness and analytical depth.

1. **Dual Preprocessing:**
To meet the specific requirements of different algorithms, two text cleaning pipelines were created:
* **Hard Preprocessing (for VADER):** Application of *Stemming*, *Lemmatization*, and *Stopwords* removal. This "classic" approach is optimized for lexical dictionaries.
* **Soft Preprocessing (for BERT/BERTopic):** Light cleaning (lowercase, URL removal) that preserves grammatical and syntactic structure, essential for contextual language models.


2. **Comparative Sentiment Analysis:**
* **VADER:** Fast *rule-based* algorithm, used as a baseline to detect explicit polarity.
* **BERT (twitter-roberta-base-sentiment):** Transformer model fine-tuned on social media, capable of grasping complex nuances like irony and implicit context.


3. **Topic Modeling (BERTopic):**
Use of **BERTopic** for unsupervised semantic clustering. The algorithm identifies the dominant themes of discussions.
* *Note:* A distinction is maintained between the **Dominant Topic (Raw)** (which includes "noise" or topic -1) and the **Reference Topic (Clean)** (the most discussed valid theme), allowing for the quantification of both dispersion and thematic focus for each community.


4. **Semantic Profiling:**
To avoid the noise of simple frequency methods (like TF-IDF), we utilized a semantic approach via **Sentence-BERT (SBERT)**. By calculating the "semantic centroid" (vector embedding) of each community's corpus, we extracted n-grams with the highest **cosine similarity** to the group's overall context, effectively capturing its core thematic identity.

The final result is a report integrating volume, sentiment, and topic metrics, offering a comprehensive view of the network's socio-semantic dynamics.

In [ ]:
# ==============================================================================
# PHASE 3: SOCIAL CONTENT ANALYSIS (SCA)
# ==============================================================================

print("\n--- Social Content Analysis (SCA) ---")

# --- 2. CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/MAGISTRALE/Social_Media/SMA_Borgia_Borserini/Datasets"
INPUT_FILE = "chat_control_sna.csv"
OUTPUT_FILE = "chat_control_FINAL.csv"
# Removed secondary summary file definition

INPUT_PATH = os.path.join(BASE_PATH, INPUT_FILE)
OUTPUT_PATH = os.path.join(BASE_PATH, OUTPUT_FILE)

# Mount Google Drive
try:
    drive.mount('/content/drive', force_remount=True)
except:
    pass

# Load Data
if not os.path.exists(INPUT_PATH):
    raise SystemExit(f"ERROR: File {INPUT_PATH} not found. Please run Phase 2 (SNA) first.")

df = pd.read_csv(INPUT_PATH)
df['comment_body'] = df['comment_body'].fillna("").astype(str)
print(f"[DATA] Loaded {len(df)} comments.")

# --- 3. COMMUNITY DEFINITION ---
print("\n[PRE-PROCESSING] Grouping Communities...")
comm_sizes = df[['comment_author', 'community']].drop_duplicates()['community'].value_counts()

# Select Top 10 Communities
top_communities = comm_sizes.head(10).index.tolist()

def label_group(comm_id):
    if comm_id in top_communities:
        return f"Comm_{int(comm_id)}"
    else:
        return "Minor_Groups"

df['community_group'] = df['community'].apply(label_group)


# --- 4. PRE-PROCESSING (Dual Pipeline) ---

# A. "Soft" Preprocessing (For BERT/BERTopic - Preserves context)
def clean_text_soft(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

df['clean_text'] = df['comment_body'].apply(clean_text_soft)

# B. "Hard" Preprocessing (For VADER - Lab requirement)
print("[PRE-PROCESSING] Applying Stemming, Lemmatization, and Stopwords for VADER...")
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')
    nltk.download('wordnet')
    nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def strict_preprocessing(text):
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    words = text.split()
    words = [w for w in words if w not in stop_words] # Remove stopwords
    words = [lemmatizer.lemmatize(w) for w in words] # Lemmatization
    return " ".join(words)

df['text_classic_proc'] = df['clean_text'].apply(strict_preprocessing)
print("[DATA] Preprocessing complete.")


# --- 5. SENTIMENT ANALYSIS: VADER ---
print("\n[SENTIMENT] Running VADER (on preprocessed text)...")
analyzer = SentimentIntensityAnalyzer()
df['vader_score'] = [analyzer.polarity_scores(t)['compound'] for t in tqdm(df['text_classic_proc'], desc="VADER")]


# --- 6. SENTIMENT ANALYSIS: BERT ---
print("\n[SENTIMENT] Running BERT (on full text)...")
MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def get_bert_batch(texts, batch_size=32):
    res_scores = []
    res_labels = []
    for i in tqdm(range(0, len(texts), batch_size), desc="BERT Batching"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad(): outputs = model(**inputs)
        probs = softmax(outputs.logits.detach().cpu().numpy(), axis=1)
        for p in probs:
            compound = p[2] - p[0]
            label_idx = np.argmax(p)
            label_str = 'negative' if label_idx == 0 else ('neutral' if label_idx == 1 else 'positive')
            res_scores.append(compound)
            res_labels.append(label_str)
    return res_scores, res_labels

scores, labels = get_bert_batch(df['clean_text'].tolist())
df['bert_score'] = scores
df['bert_label'] = labels


# --- 7. TOPIC MODELING (BERTopic) ---
print("\n[TOPIC] Running BERTopic...")

vectorizer_model = CountVectorizer(stop_words="english", min_df=10)

topic_model = BERTopic(
    language="english",
    vectorizer_model=vectorizer_model,
    representation_model=KeyBERTInspired(),
    min_topic_size=15,
    verbose=True
)

topics, probs = topic_model.fit_transform(df['clean_text'].tolist())
df['topic_id'] = topics

topic_info = topic_model.get_topic_info()

# Correct Topic Mapping (Prefix ID to Name)
topic_map = {}
for t_id, name in zip(topic_info.Topic, topic_info.Name):
    keywords = [word[0] for word in topic_model.get_topic(t_id)[:3]]
    topic_map[t_id] = f"{t_id}_" + "_".join(keywords)

df['topic_name'] = df['topic_id'].map(topic_map)

print("\n" + "-"*50)
print("TOPIC MODELING RESULTS (Top 10 Themes)")
print("-"*50)
print(topic_info[['Topic', 'Count', 'Name']].head(12).to_string(index=False))
print("-"*50)


# --- 8. DISTINCTIVE SEMANTIC KEYWORDS ---
print("\n[DIFFERENTIATION] Extracting Semantic Keywords...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def extract_semantic_keywords(text_list, top_n=None):
    if not text_list or len(text_list) < 5: return ["N/A"]
    full_text = " ".join(text_list)
    try:
        vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words="english")
        vectorizer.fit([full_text])
        words = vectorizer.get_feature_names_out()
    except: return ["N/A"]

    if len(words) == 0: return ["N/A"]

    doc_embedding = embedder.encode([full_text])
    word_embeddings = embedder.encode(words)
    distances = cosine_similarity(doc_embedding, word_embeddings)

    if top_n: top_indices = distances[0].argsort()[-top_n:][::-1]
    else: top_indices = distances[0].argsort()[::-1]

    return [words[i] for i in top_indices]

community_texts = df.groupby('community_group')['clean_text'].apply(list)
keywords_map_report = {}
keywords_map_single = {}
keywords_map_all = {}

for group_name, texts in tqdm(community_texts.items(), desc="Semantic Extraction"):
    if group_name == "Minor_Groups": # Skip minor groups
        keywords_map_report[group_name] = "Mixed"
        continue

    keywords_list = extract_semantic_keywords(texts, top_n=None)
    keywords_map_report[group_name] = ", ".join(keywords_list[:4])
    keywords_map_single[group_name] = keywords_list[0] if keywords_list else "N/A"
    keywords_map_all[group_name] = ", ".join(keywords_list)

df['semantic_keyword'] = df['community_group'].map(keywords_map_single)
df['community_all_keywords'] = df['community_group'].map(keywords_map_all)


# --- 9. COMPARATIVE ANALYSIS (Console Report Only) ---
print("\n" + "="*60)
print("SENTIMENT ANALYSIS: FINAL REPORT")
print("="*60)

# A. Global Stats
global_bert_mean = df['bert_score'].mean()
global_vader_mean = df['vader_score'].mean()
print(f"\n[GLOBAL LEVEL]")
print(f"- Sentiment BERT:  {global_bert_mean:.4f}")
print(f"- Sentiment VADER: {global_vader_mean:.4f}")

# B. Local Stats (Functions for Topics with Percentages)
def get_real_winner_stats(series):
    if series.empty: return "Mix"
    counts = series.value_counts(normalize=True)
    return f"{counts.index[0]} ({counts.iloc[0]*100:.1f}%)"

def get_clean_topic_stats(series):
    valid_topics = series[~series.str.startswith("-1_")]
    if valid_topics.empty: return "Noise Only"
    counts = valid_topics.value_counts(normalize=True)
    return f"{counts.index[0]} ({counts.iloc[0]*100:.1f}%)"

print(f"\n[LOCAL LEVEL] (Top Communities)")
summary = df.groupby('community_group').agg({
    'comment_id': 'count',
    'bert_score': 'mean',
    'vader_score': 'mean',
    'topic_name': [get_real_winner_stats, get_clean_topic_stats]
}).sort_values(('comment_id', 'count'), ascending=False)

summary.columns = ['num_comments', 'avg_bert', 'avg_vader', 'Dominant_Topic_Raw', 'Reference_Topic_Clean']

# Filter Minor Groups
summary = summary[summary.index != "Minor_Groups"]

summary['semantic_keywords'] = summary.index.map(keywords_map_report)
summary['role'] = summary['avg_bert'].apply(lambda x: "Positive" if x > 0.05 else ("Negative" if x < -0.05 else "Neutral"))

# Display Table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(summary[['num_comments', 'avg_bert', 'avg_vader', 'role', 'Dominant_Topic_Raw', 'Reference_Topic_Clean', 'semantic_keywords']])


# --- 10. SAVING ---
print("\n[SAVING] Writing main file to Drive...")
df.to_csv(OUTPUT_PATH, index=False)
print(f"File saved successfully: {OUTPUT_PATH}")


--- Social Content Analysis (SCA) ---
Mounted at /content/drive
[DATA] Loaded 6944 comments.

[PRE-PROCESSING] Grouping Communities...
[PRE-PROCESSING] Applying Stemming, Lemmatization, and Stopwords for VADER...
[DATA] Preprocessing complete.

[SENTIMENT] Running VADER (on preprocessed text)...


VADER:   0%|          | 0/6944 [00:00<?, ?it/s]


[SENTIMENT] Running BERT (on full text)...


BERT Batching:   0%|          | 0/217 [00:00<?, ?it/s]

2026-01-06 12:28:04,760 - BERTopic - Embedding - Transforming documents to embeddings.


...

Batches:   0%|          | 0/217 [00:00<?, ?it/s]

2026-01-06 12:28:10,868 - BERTopic - Embedding - Completed ✓
2026-01-06 12:28:10,869 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-06 12:28:21,758 - BERTopic - Dimensionality - Completed ✓
2026-01-06 12:28:21,759 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-06 12:28:22,023 - BERTopic - Cluster - Completed ✓
2026-01-06 12:28:22,029 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-06 12:28:22,921 - BERTopic - Representation - Completed ✓



--------------------------------------------------
TOPIC MODELING RESULTS (Top 10 Themes)
--------------------------------------------------
 Topic  Count                                              Name
    -1   2520               -1_surveillance_privacy_eu_security
     0    564                 0_policies_policy_citizens_rights
     1    290      1_surveillance_government_politicians_social
     2    166              2_privacy_freedoms_anonymous_private
     3    146          3_government_politicians_article_citizen
     4    143                     4_stupid_isnt_really_actually
     5    140               5_privacy_abuse_consequences_safety
     6    131           6_oppose_opposition_ukraine_politicians
     7    112          7_backdoors_backdoor_regulation_proposal
     8    103           8_governments_opposition_opposed_refuse
     9     98 9_governments_politicians_democracy_conservatives
    10     90           10_eventually_government_happening_note
--------------------------

Semantic Extraction: 0it [00:00, ?it/s]


SENTIMENT ANALYSIS: FINAL REPORT

[GLOBAL LEVEL]
- Sentiment BERT:  -0.4106
- Sentiment VADER: 0.0505

[LOCAL LEVEL] (Top Communities)
                 num_comments  avg_bert  avg_vader      role                  Dominant_Topic_Raw                          Reference_Topic_Clean                                  semantic_keywords
community_group                                                                                                                                                                                   
Comm_20                   693 -0.429775   0.021192  Negative  -1_surveillance_privacy_eu (39.2%)           2_privacy_freedoms_anonymous (11.4%)  china access, totalitarian surveillance, chine...
Comm_3                    400 -0.432887   0.063871  Negative  -1_surveillance_privacy_eu (38.2%)             0_policies_policy_citizens (10.9%)  eu surveillance, eu spying, surveillance censo...
Comm_22                   350 -0.420943   0.027595  Negative  -1_surveillance_privac

### **Results SCA**

### 1. Global Sentiment Landscape (Polarization)
* **BERT Score:** -0.4106 (Strongly Negative)
* **VADER Score:** 0.0505 (Neutral/Slightly Positive)
* **Interpretation:** The discrepancy between the two models confirms the limitations of traditional methods.
    * **VADER Failure:** The lexicon-based model fails to grasp context. It likely misinterprets terms like "safety," "rights," and "protection" (frequent in Topic 0) as positive, missing the fact that users are attacking the *legislation* ostensibly named after these concepts.
    * **BERT Accuracy:** The Transformer model correctly captures the underlying tone of anger, fear, and institutional distrust. A global average of -0.41 indicates a discussion pervaded by hostility, where even "neutral" words are used in a critical context.

### 2. Thematic Landscape (Topic Modeling)

* **Dominant Theme (Topic 0 - 564 docs):** *Policies, Policy, Citizens, Rights*. The core of the debate is rational and legalistic, focusing on the specific legislative mechanisms and the erosion of citizens' rights.
* **Institutional Distrust (Topic 1 - 290 docs):** *Surveillance, Government, Politicians*. This cluster shifts focus from the law to the actors, expressing deep suspicion towards politicians and government transparency.
* **Privacy & Anonymity (Topic 2 - 166 docs):** *Privacy, Freedoms, Anonymous, Private*. A distinct thematic group focusing on the loss of anonymity and the "right to be private" as a fundamental freedom, distinct from the technical safety arguments.
* **Safety vs. Abuse (Topic 5 - 140 docs):** *Privacy, Abuse, Consequences, Safety*. Users actively deconstruct the "Child Safety" narrative, arguing that the consequences of potential abuse of power outweigh the promised safety benefits.

### 3. Community Profiling (Semantic Differentiation)

The analysis of the top communities reveals that while the sentiment is uniformly Negative (BERT scores range from -0.38 to -0.43), the *nature* of their opposition varies significantly:

* **The "Dystopian" Cluster (Comm_20):**
    * *Keywords:* `china access`, `totalitarian surveillance`, `chinese style`.
    * *Focus:* The largest active community (693 comments) frames the discussion in catastrophic terms. They explicitly compare the EU proposal to the Chinese Social Credit System and define the legislation as "totalitarian," fearing a complete collapse of Western democratic values.

* **The "Orwellian" Cluster (Comm_7):**
    * *Keywords:* `1984 eu`, `ussr 1984`, `arent 1984`, `1984 warning`.
    * *Focus:* A highly specific ideological group that frames the legislation strictly through the lens of George Orwell's *1984* and comparisons to the USSR, treating the bill not as a policy change but as a regime change.

* **The "Danish/Political" Cluster (Comm_0):**
    * *Keywords:* `laws denmark`, `protests danes`, `eu law`.
    * *Focus:* A unique, highly localized reaction. This community specifically targets Denmark (likely due to a Danish minister's strong support for the bill), showing that the debate is not just abstract but tied to specific national actors and protests.

* **The "Eurosceptic" Cluster (Comm_1):**
    * *Keywords:* `euroscepticism`, `euroskeptic politicians`, `destroy eu`.
    * *Focus:* For this group, "Chat Control" is merely a symptom. Their opposition is rooted in a broader desire to dismantle the European Union ("destroy eu"), using this legislation as proof that the EU project is inherently malicious.